# Background Removal: Method Comparison

## Imports

In [ ]:
import sys
from pathlib import Path

import cv2
import numpy as np
import matplotlib.pyplot as plt

projectRoot = Path(".").resolve()
if str(projectRoot) not in sys.path:
    sys.path.insert(0, str(projectRoot))

from bg_subtract import subtract_global_median
from bg_subtract_rolling import subtract_rolling_median
from bg_methods import (
    subtractBaselineMedian,
    subtractGlobalAverage,
    subtractSlidingAverage,
)

%matplotlib inline
plt.rcParams["figure.dpi"] = 120
plt.rcParams["image.cmap"] = "gray"

print("Imports OK")

## Select Input Video

In [ ]:
# ── EDIT THIS ──────────────────────────────────────────────────────
videoPath = Path("1/1.avi")
# ──────────────────────────────────────────────────────────────────

videoPath = videoPath.resolve()

if not videoPath.exists():
    raise FileNotFoundError(f"Video not found: {videoPath}")

resultsDir = Path("Results").resolve()
resultsDir.mkdir(parents=True, exist_ok=True)

print(f"Video:   {videoPath}")
print(f"Results: {resultsDir}")

## Video Metadata

In [ ]:
cap = cv2.VideoCapture(str(videoPath))
if not cap.isOpened():
    raise RuntimeError(f"OpenCV cannot open: {videoPath}")

fps = cap.get(cv2.CAP_PROP_FPS)
frameCount = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
frameWidth = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frameHeight = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
cap.release()

print(f"FPS:          {fps}")
print(f"Frame count:  {frameCount}")
print(f"Resolution:   {frameWidth} x {frameHeight}")
if fps > 0:
    print(f"Duration:     {frameCount / fps:.2f} s")

## Original Frame

In [ ]:
sampleIndex = frameCount // 2

cap = cv2.VideoCapture(str(videoPath))
cap.set(cv2.CAP_PROP_POS_FRAMES, sampleIndex)
ok, sampleBGR = cap.read()
cap.release()

if not ok:
    raise RuntimeError(f"Could not read frame {sampleIndex}")

sampleGray = cv2.cvtColor(sampleBGR, cv2.COLOR_BGR2GRAY)

fig, ax = plt.subplots(figsize=(8, 6))
ax.imshow(sampleGray, cmap="gray")
ax.set_title(f"Original grayscale, frame {sampleIndex}")
ax.axis("off")
plt.tight_layout()
plt.show()

## Baseline: Brightness Flattening + Global Temporal Median

In [ ]:
baselineOutPath = resultsDir / f"{videoPath.stem}_baseline.mp4"

baselineBg, baselineOutPath = subtractBaselineMedian(
    str(videoPath),
    outPath=str(baselineOutPath),
)

## Global Temporal Median

In [ ]:
globalMedianOutPath = resultsDir / f"{videoPath.stem}_global_median.mp4"

globalMedianBg, globalMedianOutPath = subtract_global_median(
    str(videoPath),
    out_path=str(globalMedianOutPath),
)

## Global Frame Average

In [ ]:
globalAvgOutPath = resultsDir / f"{videoPath.stem}_global_avg.mp4"

globalAvgBg, globalAvgOutPath = subtractGlobalAverage(
    str(videoPath),
    outPath=str(globalAvgOutPath),
)

## Rolling Temporal Median

In [ ]:
rollingMedianOutPath = resultsDir / f"{videoPath.stem}_rolling_median.mp4"

rollingMedianOutPath = subtract_rolling_median(
    str(videoPath),
    out_path=str(rollingMedianOutPath),
)

## Sliding-Window Frame Average

In [ ]:
slidingAvgOutPath = resultsDir / f"{videoPath.stem}_sliding_avg.mp4"

slidingAvgOutPath = subtractSlidingAverage(
    str(videoPath),
    outPath=str(slidingAvgOutPath),
)

## Read Processed Frames

In [ ]:
def readFrameGray(videoFile, index):
    cap = cv2.VideoCapture(str(videoFile))
    cap.set(cv2.CAP_PROP_POS_FRAMES, index)
    ok, frameBGR = cap.read()
    cap.release()
    if not ok:
        raise RuntimeError(f"Could not read frame {index} from {videoFile}")
    return cv2.cvtColor(frameBGR, cv2.COLOR_BGR2GRAY)

baselineFrame      = readFrameGray(baselineOutPath, sampleIndex)
globalMedianFrame  = readFrameGray(globalMedianOutPath, sampleIndex)
globalAvgFrame     = readFrameGray(globalAvgOutPath, sampleIndex)
rollingMedianFrame = readFrameGray(rollingMedianOutPath, sampleIndex)
slidingAvgFrame    = readFrameGray(slidingAvgOutPath, sampleIndex)

print(f"Read frame {sampleIndex} from all five outputs.")

## Side-by-Side Comparison

In [ ]:
panels = [
    (sampleGray,        "Original"),
    (baselineFrame,     "Baseline\n(flatten + global median)"),
    (globalMedianFrame, "Global median"),
    (globalAvgFrame,    "Global average"),
    (rollingMedianFrame,"Rolling median"),
    (slidingAvgFrame,   "Sliding-window average"),
]

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.ravel()

for ax, (img, title) in zip(axes, panels):
    ax.imshow(img, cmap="gray")
    ax.set_title(title, fontsize=12)
    ax.axis("off")

fig.suptitle(f"Frame {sampleIndex}, {videoPath.name}", fontsize=14)
plt.tight_layout()
plt.show()

## Intensity Histograms

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 8))
axes = axes.ravel()

for ax, (img, label) in zip(axes, panels):
    ax.hist(img.ravel(), bins=256, range=(0, 255), color="steelblue", alpha=0.8)
    ax.set_title(label, fontsize=10)
    ax.set_xlabel("Pixel intensity")
    ax.set_ylabel("Count")

fig.suptitle(f"Intensity histograms, frame {sampleIndex}", fontsize=14)
plt.tight_layout()
plt.show()

## Background Images

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

bgPanels = [
    (baselineBg,     "Baseline (flattened global median)"),
    (globalMedianBg, "Global median"),
    (globalAvgBg,    "Global average"),
]

for ax, (img, title) in zip(axes, bgPanels):
    ax.imshow(img, cmap="gray")
    ax.set_title(title)
    ax.axis("off")

plt.tight_layout()
plt.show()